# Phase 3: Model Training and Evaluation

This notebook trains Logistic Regression, KNN, and SVM classifiers using Python only.

In [1]:
import sys
from pathlib import Path
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

project_root = Path.cwd().parent if Path.cwd().name == 'Phase_3' else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from main import (
    build_models,
    build_target_labels,
    choose_features,
    engineer_features,
    load_and_clean_data,
    select_feature_columns,
)


In [2]:
df = load_and_clean_data()
df = build_target_labels(df)
df = engineer_features(df)

feature_columns = select_feature_columns(df)
model_df = df[feature_columns + ['target_aqi_category', 'datetime']].copy()
model_df = model_df.sort_values('datetime').reset_index(drop=True)

split_index = int(len(model_df) * 0.8)
train_df = model_df.iloc[:split_index].copy()
test_df = model_df.iloc[split_index:].copy()

X_train_raw = train_df[feature_columns]
y_train = train_df['target_aqi_category'].astype(str)
X_test_raw = test_df[feature_columns]
y_test = test_df['target_aqi_category'].astype(str)

selector_imputer = SimpleImputer(strategy='median')
X_train_imputed = pd.DataFrame(
    selector_imputer.fit_transform(X_train_raw),
    columns=feature_columns,
    index=X_train_raw.index,
)

selected_features, _ = choose_features(X_train_imputed, y_train, feature_columns)
X_train = X_train_raw[selected_features]
X_test = X_test_raw[selected_features]

selected_features


['PT08.S2(NMHC)',
 'C6H6(GT)',
 'PT08.S1(CO)',
 'PT08.S5(O3)',
 'CO(GT)',
 'PT08.S3(NOx)',
 'NO2(GT)',
 'PT08.S4(NO2)',
 'lag1_C6H6(GT)',
 'NOx(GT)',
 'lag1_CO(GT)',
 'lag1_NOx(GT)',
 'T',
 'RH',
 'hour']

In [3]:
results = []
for model_name, model in build_models().items():
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)
    results.append({
        'model': model_name,
        'accuracy': round(accuracy_score(y_test, predictions), 4),
        'precision_macro': round(precision_score(y_test, predictions, average='macro', zero_division=0), 4),
        'recall_macro': round(recall_score(y_test, predictions, average='macro', zero_division=0), 4),
        'f1_macro': round(f1_score(y_test, predictions, average='macro', zero_division=0), 4),
    })

pd.DataFrame(results).sort_values('f1_macro', ascending=False)


,model,accuracy,precision_macro,recall_macro,f1_macro
2,svm,0.5458,0.5173,0.5245,0.5183
1,knn,0.5169,0.4918,0.4995,0.4923
0,logistic_regression,0.5292,0.5091,0.5009,0.4825
